In [ ]:
#Section 1: Data Loading & Initial Inspection

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [16]:
# Load the dataset
df = pd.read_csv("data/raw/us_pollution.csv")

In [17]:
# Filter to only keep data from 2022 onwards to meet row limit
df = df[df["Date"] >= "2022-01-01"]
df = df.reset_index(drop=True)

print("New shape:", df.shape)

New shape: (44726, 22)


In [18]:
# Basic inspection
print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names and data types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nBasic statistics:")
print(df.describe())

Shape: (44726, 22)

First 5 rows:
   Unnamed: 0        Date                                Address    State  \
0      620688  2022-01-01  NO. B'HAM,SOU R.R., 3009 28TH ST. NO.  Alabama   
1      620689  2022-01-02  NO. B'HAM,SOU R.R., 3009 28TH ST. NO.  Alabama   
2      620690  2022-01-04  NO. B'HAM,SOU R.R., 3009 28TH ST. NO.  Alabama   
3      620691  2022-01-05  NO. B'HAM,SOU R.R., 3009 28TH ST. NO.  Alabama   
4      620692  2022-01-06  NO. B'HAM,SOU R.R., 3009 28TH ST. NO.  Alabama   

      County        City   O3 Mean  O3 1st Max Value  O3 1st Max Hour  O3 AQI  \
0  Jefferson  Birmingham  0.024765             0.026               21      24   
1  Jefferson  Birmingham  0.017824             0.025               23      23   
2  Jefferson  Birmingham  0.025353             0.030               10      28   
3  Jefferson  Birmingham  0.012563             0.034                8      31   
4  Jefferson  Birmingham  0.023941             0.028               18      26   

   ...  CO 1st M

In [19]:
# pd.Series and pd.DataFrame 
pollutants = pd.Series(["NO2", "O3", "SO2", "CO"], name="Tracked Pollutants")
print("\nPollutants tracked in this dataset:")
print(pollutants)

health_info = pd.DataFrame({
    "Pollutant": ["NO2", "O3", "SO2", "CO"],
    "Health Effect": [
        "Respiratory inflammation",
        "Lung tissue damage",
        "Throat and eye irritation",
        "Reduces oxygen in bloodstream"
    ],
    "EPA Limit (AQI)": [100, 100, 100, 100]
})
print("\nPollutant health reference table:")
print(health_info)


Pollutants tracked in this dataset:
0    NO2
1     O3
2    SO2
3     CO
Name: Tracked Pollutants, dtype: object

Pollutant health reference table:
  Pollutant                  Health Effect  EPA Limit (AQI)
0       NO2       Respiratory inflammation              100
1        O3             Lung tissue damage              100
2       SO2      Throat and eye irritation              100
3        CO  Reduces oxygen in bloodstream              100


In [20]:
# Type conversions
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Extract month and year columns for later grouping
df["Year"]  = df["Date"].dt.year
df["Month"] = df["Date"].dt.month

# Convert AQI columns to numeric (coerce handles any bad values)
aqi_cols = ["O3 AQI", "CO AQI", "SO2 AQI", "NO2 AQI"]
for col in aqi_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Date range:", df["Date"].min(), "→", df["Date"].max())
print("dtypes after conversion:\n", df[["Date","Year","Month"] + aqi_cols].dtypes)

Date range: 2022-01-01 00:00:00 → 2023-09-30 00:00:00
dtypes after conversion:
 Date       datetime64[ns]
Year                int32
Month               int32
O3 AQI              int64
CO AQI            float64
SO2 AQI           float64
NO2 AQI             int64
dtype: object


In [24]:
# Missing value handling 
print("Missing values BEFORE:\n", df.isnull().sum()[df.isnull().sum() > 0])

# Strategy 1: Drop rows where ALL AQI columns are null (unusable rows)
df.dropna(subset=aqi_cols, how="all", inplace=True)

# Strategy 2: Fill remaining AQI nulls with column median
for col in aqi_cols:
    df[col] = df[col].fillna(df[col].median())

print("\nMissing values AFTER:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("Shape after cleaning:", df.shape)

Missing values BEFORE:
 Series([], dtype: int64)

Missing values AFTER:
 Series([], dtype: int64)
Shape after cleaning: (44726, 26)


In [22]:
# Derive new columns

# np.where: classify O3 AQI as Good / Moderate / Unhealthy
df["O3_Category"] = np.where(df["O3 AQI"] <= 50, "Good",
                    np.where(df["O3 AQI"] <= 100, "Moderate", "Unhealthy"))

# .apply: compute an average AQI score across all 4 pollutants
df["Avg_AQI"] = df[aqi_cols].apply(lambda row: row.mean(), axis=1)

print(df[["Date", "State", "O3 AQI", "O3_Category", "Avg_AQI"]].head(10))
print("\nO3 Category distribution:\n", df["O3_Category"].value_counts())

        Date    State  O3 AQI O3_Category  Avg_AQI
0 2022-01-01  Alabama      24        Good     6.50
1 2022-01-02  Alabama      23        Good     7.25
2 2022-01-04  Alabama      28        Good     9.75
3 2022-01-05  Alabama      31        Good    16.25
4 2022-01-06  Alabama      26        Good    13.50
5 2022-01-07  Alabama      23        Good    11.75
6 2022-01-08  Alabama      31        Good    15.00
7 2022-01-09  Alabama      24        Good     9.75
8 2022-01-11  Alabama      29        Good    13.75
9 2022-01-12  Alabama      30        Good    18.25

O3 Category distribution:
 O3_Category
Good         38377
Moderate      5528
Unhealthy      821
Name: count, dtype: int64


In [23]:
# Indexing examples

# .loc — label-based: filter to Florida rows
florida = df.loc[df["State"] == "Florida"]
print("Florida rows:", florida.shape)
print(florida[["Date","City","O3 AQI","NO2 AQI"]].head())

# .iloc — position-based: first 5 rows, first 6 columns
print("\nFirst 5 rows, first 6 cols (iloc):")
print(df.iloc[:5, :6])

Florida rows: (373, 26)
            Date   City  O3 AQI  NO2 AQI
10974 2022-06-01  Davie      36        8
10975 2022-06-02  Davie      35       16
10976 2022-06-03  Davie      18        8
10977 2022-06-04  Davie      21        3
10978 2022-06-05  Davie      31        6

First 5 rows, first 6 cols (iloc):
   Unnamed: 0       Date                                Address    State  \
0      620688 2022-01-01  NO. B'HAM,SOU R.R., 3009 28TH ST. NO.  Alabama   
1      620689 2022-01-02  NO. B'HAM,SOU R.R., 3009 28TH ST. NO.  Alabama   
2      620690 2022-01-04  NO. B'HAM,SOU R.R., 3009 28TH ST. NO.  Alabama   
3      620691 2022-01-05  NO. B'HAM,SOU R.R., 3009 28TH ST. NO.  Alabama   
4      620692 2022-01-06  NO. B'HAM,SOU R.R., 3009 28TH ST. NO.  Alabama   

      County        City  
0  Jefferson  Birmingham  
1  Jefferson  Birmingham  
2  Jefferson  Birmingham  
3  Jefferson  Birmingham  
4  Jefferson  Birmingham  


In [25]:
# Multi-column aggregation: mean and count of Avg_AQI by State + Year
state_year = (df.groupby(["State", "Year"])["Avg_AQI"]
                .agg(["mean", "count"])
                .rename(columns={"mean": "Mean_AQI", "count": "Days"})
                .reset_index())

print(state_year.head(10))

         State  Year   Mean_AQI  Days
0      Alabama  2022  15.484058   345
1      Alabama  2023  17.001404   178
2      Arizona  2022  19.267348  1052
3      Arizona  2023  20.788628   576
4     Arkansas  2022  14.018169   344
5   California  2022  16.620967  7190
6   California  2023  14.521133  3336
7     Colorado  2022  19.674250   967
8     Colorado  2023  20.531631   656
9  Connecticut  2022  15.852227   247


In [26]:
# Pivot: Mean AQI by State × Year
pivot = state_year.pivot_table(
    index="State",
    columns="Year",
    values="Mean_AQI"
)
print("Mean AQI by State × Year:")
print(pivot.round(1).head(10))

Mean AQI by State × Year:
Year                  2022  2023
State                           
Alabama               15.5  17.0
Arizona               19.3  20.8
Arkansas              14.0   NaN
California            16.6  14.5
Colorado              19.7  20.5
Connecticut           15.9  15.5
Delaware              15.2   NaN
District Of Columbia  15.4  17.2
Florida               11.0  14.8
Georgia               15.0  16.7


In [27]:
import os
os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/us_pollution_cleaned.csv", index=False)
print("Saved to data/processed/us_pollution_cleaned.csv")

Saved to data/processed/us_pollution_cleaned.csv
